# Suraya Car — Model Training (Colab)
Trains both models used by the Suraya Car app and exports them in the exact format the app loads:

| File | Model | Target |
|---|---|---|
| `yolov8_detector.pt` | YOLOv8m damage detector (7 classes) | mAP@0.5 >= 0.80 |
| `efficientnetv2.pth` | EfficientNetV2-S severity classifier (3 classes) | val accuracy >= 89% |

**Before running:** Runtime > Change runtime type > **T4 GPU** (or better: A100/L4).
Total time: ~60-90 min on T4. Run cells top to bottom.

In [ ]:
#@title CELL 1 — Install dependencies (~2 min)
!pip uninstall -y wandb -q
!pip install -q "ultralytics>=8.2" timm albumentations roboflow grad-cam scikit-learn

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable GPU runtime!")
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU" 

In [ ]:
#@title CELL 2 — Download dataset from Roboflow
# Get a FREE API key: https://app.roboflow.com > Settings > API Keys
ROBOFLOW_API_KEY = ""  #@param {type:"string"}

from roboflow import Roboflow
from pathlib import Path

assert ROBOFLOW_API_KEY, "Paste your Roboflow API key above (free at app.roboflow.com)"
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("capstone-nh0nc").project("car-damage-detection-t0g92")
dataset = project.version(1).download("yolov8")
DATASET_PATH = Path(dataset.location)
print("Dataset at:", DATASET_PATH)

# If the dataset above is unavailable, search "car damage detection" on
# https://universe.roboflow.com, pick one with 7 classes (bonnet, bumper,
# dickey, door, fender, light, windshield) and replace workspace/project above.

In [ ]:
#@title CELL 3 — Inspect dataset & fix data.yaml
import yaml, glob

yaml_path = DATASET_PATH / "data.yaml"
cfg = yaml.safe_load(open(yaml_path))
print("Classes:", cfg["names"])
NUM_CLASSES = len(cfg["names"])

n_train = len(glob.glob(str(DATASET_PATH/"train/images/*")))
n_val   = len(glob.glob(str(DATASET_PATH/"valid/images/*")))
n_test  = len(glob.glob(str(DATASET_PATH/"test/images/*")))
print(f"train={n_train}  val={n_val}  test={n_test}")

# absolute paths so ultralytics finds them
cfg["path"] = str(DATASET_PATH)
cfg["train"] = "train/images"; cfg["val"] = "valid/images"; cfg["test"] = "test/images"
yaml.safe_dump(cfg, open(yaml_path, "w"))
print("data.yaml updated")

In [ ]:
#@title CELL 4 — Train YOLOv8m detector (~35-50 min on T4)
from ultralytics import YOLO

det = YOLO("yolov8m.pt")
det_results = det.train(
    data=str(DATASET_PATH/"data.yaml"),
    epochs=80, patience=20, imgsz=640, batch=16,
    optimizer="AdamW", lr0=1e-3, lrf=0.01, cos_lr=True,
    mosaic=1.0, mixup=0.15, copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10, translate=0.1, scale=0.5, fliplr=0.5,
    warmup_epochs=3, weight_decay=5e-4,
    project="runs", name="detector", exist_ok=True, verbose=False,
)
print("Detector training done")

In [ ]:
#@title CELL 5 — Evaluate detector (must be >= 0.80 mAP@0.5)
from ultralytics import YOLO
BEST_DET = "runs/detector/weights/best.pt"
det = YOLO(BEST_DET)
metrics = det.val(data=str(DATASET_PATH/"data.yaml"), split="test", verbose=False)
map50 = float(metrics.box.map50)
print(f"mAP@0.5      = {map50:.3f}")
print(f"mAP@0.5:0.95 = {float(metrics.box.map):.3f}")
if map50 < 0.80:
    print("WARNING: below 0.80 - consider more epochs (set epochs=120 in Cell 4) or a larger dataset version")
else:
    print("Detector target reached")

In [ ]:
#@title CELL 6 — Build severity dataset (crops labeled by damage size)
# Severity ground truth is derived from the relative area of each damage box:
# small surface damage = minor, mid-size = moderate, large deformation = severe.
import cv2, shutil, random
from pathlib import Path
import numpy as np

SEVERITY_DIR = Path("/content/severity")
shutil.rmtree(SEVERITY_DIR, ignore_errors=True)
for split in ["train","val"]:
    for c in ["minor","moderate","severe"]:
        (SEVERITY_DIR/split/c).mkdir(parents=True, exist_ok=True)

def severity_from_area(rel_area):
    if rel_area < 0.030: return "minor"
    if rel_area < 0.110: return "moderate"
    return "severe"

random.seed(42)
counts = {"minor":0,"moderate":0,"severe":0}
for src_split, dst_split in [("train","train"), ("valid","val"), ("test","val")]:
    img_dir = DATASET_PATH/src_split/"images"
    lbl_dir = DATASET_PATH/src_split/"labels"
    if not img_dir.exists(): continue
    for img_path in img_dir.glob("*"):
        lbl_path = lbl_dir/(img_path.stem + ".txt")
        if not lbl_path.exists(): continue
        img = cv2.imread(str(img_path))
        if img is None: continue
        H, W = img.shape[:2]
        for i, line in enumerate(open(lbl_path)):
            p = line.split()
            if len(p) < 5: continue
            _, xc, yc, w, h = map(float, p[:5])
            rel_area = w * h
            sev = severity_from_area(rel_area)
            x1 = int(max(0,(xc-w/2)*W)-8); y1 = int(max(0,(yc-h/2)*H)-8)
            x2 = int(min(W,(xc+w/2)*W)+8); y2 = int(min(H,(yc+h/2)*H)+8)
            crop = img[max(0,y1):y2, max(0,x1):x2]
            if crop.size == 0 or min(crop.shape[:2]) < 24: continue
            out = SEVERITY_DIR/dst_split/sev/f"{img_path.stem}_{i}.jpg"
            cv2.imwrite(str(out), crop)
            counts[sev] += 1
print("Crops per class:", counts)

In [ ]:
#@title CELL 7 — Train EfficientNetV2-S severity classifier (~20-30 min)
import timm, torch, torch.nn as nn, torch.optim as optim, cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter

device = "cuda"
SEVERITY_CLASSES = ["minor", "moderate", "severe"]

train_tfm = A.Compose([
    A.Resize(224,224), A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5), A.HueSaturationValue(p=0.3),
    A.GaussNoise(p=0.2), A.Rotate(limit=15, p=0.4),
    A.CoarseDropout(max_holes=4, max_height=28, max_width=28, p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()])
val_tfm = A.Compose([
    A.Resize(224,224),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()])

class SeverityDataset(Dataset):
    def __init__(self, root, tfm):
        self.tfm = tfm
        self.samples = []
        for i, c in enumerate(SEVERITY_CLASSES):
            for f in (Path(root)/c).glob("*.jpg"):
                self.samples.append((str(f), i))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        return self.tfm(image=img)["image"], label

train_ds = SeverityDataset("/content/severity/train", train_tfm)
val_ds   = SeverityDataset("/content/severity/val",   val_tfm)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f"train={len(train_ds)}  val={len(val_ds)}")

# class weights for imbalance
cnt = Counter(l for _, l in train_ds.samples)
weights = torch.tensor([len(train_ds)/(3*cnt[i]) for i in range(3)], dtype=torch.float32).to(device)
print("class weights:", weights.cpu().numpy().round(2))

# IMPORTANT: identical architecture to the app (app/ml/pipeline.py)
class SeverityClassifier(nn.Module):
    def __init__(self, nc=3):
        super().__init__()
        self.backbone = timm.create_model("efficientnetv2_s", pretrained=True,
                                          num_classes=0, global_pool="avg")
        fdim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(fdim,512), nn.ReLU(),
            nn.BatchNorm1d(512), nn.Dropout(0.2), nn.Linear(512,3))
    def forward(self, x): return self.head(self.backbone(x))

clf = SeverityClassifier().to(device)
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
EPOCHS = 40
optimizer = optim.AdamW(clf.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

for p in clf.backbone.parameters(): p.requires_grad = False  # freeze 5 epochs

def run_epoch(loader, train=True):
    clf.train() if train else clf.eval()
    loss_sum = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                out = clf(imgs); loss = criterion(out, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            loss_sum += loss.item()
            correct += out.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    return loss_sum/len(loader), 100.*correct/total

best_acc = 0.0
CKPT = "/content/severity_best.pth"
for epoch in range(EPOCHS):
    if epoch == 5:
        for p in clf.backbone.parameters(): p.requires_grad = True
        optimizer = optim.AdamW(clf.parameters(), lr=1e-4, weight_decay=1e-4)
        print("backbone unfrozen")
    tl, ta = run_epoch(train_dl, True)
    vl, va = run_epoch(val_dl, False)
    scheduler.step()
    if va > best_acc:
        best_acc = va
        torch.save({"model": clf.state_dict(), "acc": va, "classes": SEVERITY_CLASSES}, CKPT)
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"[{epoch+1:02d}/{EPOCHS}] train {ta:.1f}% | val {va:.1f}% | best {best_acc:.1f}%")

print(f"BEST VAL ACCURACY: {best_acc:.2f}%")

In [ ]:
#@title CELL 8 — Auto fine-tune if below 89%
TARGET = 89.0
if best_acc < TARGET:
    print(f"{best_acc:.1f}% < {TARGET}% - fine-tuning 20 more epochs at low LR...")
    ckpt = torch.load(CKPT); clf.load_state_dict(ckpt["model"])
    optimizer = optim.AdamW(clf.parameters(), lr=2e-5, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-7)
    for epoch in range(20):
        tl, ta = run_epoch(train_dl, True)
        vl, va = run_epoch(val_dl, False)
        scheduler.step()
        if va > best_acc:
            best_acc = va
            torch.save({"model": clf.state_dict(), "acc": va, "classes": SEVERITY_CLASSES}, CKPT)
        print(f"  ft[{epoch+1:02d}/20] val {va:.1f}% | best {best_acc:.1f}%")
print(f"FINAL BEST: {best_acc:.2f}%  ({'TARGET REACHED' if best_acc >= TARGET else 'below target - see tips at the bottom'})")

In [ ]:
#@title CELL 9 — Confusion matrix & per-class report
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

ckpt = torch.load(CKPT); clf.load_state_dict(ckpt["model"]); clf.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in val_dl:
        out = clf(imgs.to(device))
        y_pred += out.argmax(1).cpu().tolist(); y_true += labels.tolist()
print(classification_report(y_true, y_pred, target_names=SEVERITY_CLASSES, digits=3))
cm = confusion_matrix(y_true, y_pred)
plt.imshow(cm, cmap="Blues"); plt.xticks(range(3), SEVERITY_CLASSES); plt.yticks(range(3), SEVERITY_CLASSES)
for i in range(3):
    for j in range(3): plt.text(j, i, cm[i,j], ha="center", color="red")
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title(f"val acc {ckpt['acc']:.1f}%"); plt.show()

In [ ]:
#@title CELL 10 — Export BOTH files for the app + save to Drive
import shutil
from google.colab import drive, files

shutil.copy(BEST_DET, "/content/yolov8_detector.pt")
shutil.copy(CKPT, "/content/efficientnetv2.pth")

# sanity-check: load exactly like the app does
from ultralytics import YOLO
_d = YOLO("/content/yolov8_detector.pt"); print("detector classes:", _d.names)
_c = torch.load("/content/efficientnetv2.pth", map_location="cpu")
_m = SeverityClassifier(); _m.load_state_dict(_c["model"]); print(f"classifier OK - acc {_c['acc']:.1f}%")

drive.mount("/content/drive")
out = "/content/drive/MyDrive/SurayaCar_models"
import os; os.makedirs(out, exist_ok=True)
shutil.copy("/content/yolov8_detector.pt", out)
shutil.copy("/content/efficientnetv2.pth", out)
print("Saved to Google Drive:", out)

files.download("/content/yolov8_detector.pt")
files.download("/content/efficientnetv2.pth")

## After training
1. Put both downloaded files into `Desktop\VehicleDamageAI\models\` on your laptop (exact names: `yolov8_detector.pt`, `efficientnetv2.pth`).
2. Restart the server. `http://localhost:8000/health` must show `"model_source": "trained"`.

## If severity accuracy stays below 89%
- Run Cell 4 with `epochs=120` (better crops come from a better detector's labels).
- In Cell 6, adjust thresholds `0.030 / 0.110` — check the class counts are balanced (no class < 15% of total).
- Use a bigger dataset version on Roboflow Universe (search "car damage detection", pick 5,000+ images).
- Switch runtime to A100/L4 and raise batch to 64.

## Honest note
Severity labels here are derived from damage size (area heuristic), same as the original project. The accuracy number measures agreement with that heuristic. For insurer-grade severity, you would eventually want human-labeled severity data.